In [ ]:
# @title Chapter 6 第3〜4回の学習用ベース。これを各自のGoogle Driveにコピー保存して作業開始

In [ ]:
# @title 第3〜4回でやること（本 p.195〜終わりまで）

# (1) 動画の準備
# (2) 動画からフレーム抽出 (第2回のおさらい)
# (3) 物体検出ライブラリを準備 ( 〃 )
# (4) 動画から人を検出、人数を表示 ( 〃 )
# (5) 顔の領域を抽出 (本 6-5-1)
# (6) 笑顔を検出 (本 6-5-2)
# (7) 笑顔の人数を表示 (本 6-5-3)
# (8) 全員笑顔になったら撮影 (本 6-6-1)# (9) 応用例：全員笑顔になったら動画を止めて撮影、文字と元画像 (少し大きく) を表示

In [ ]:
# @title (1) 動画の準備

# 何でもいいので、複数人の顔が写っている動画を準備します
# 例えば https://video-ac.com/video/discovery?search=笑顔%20複数人
# 本 p.198〜で使っているのは https://video-ac.com/video/17803
# 無料の会員登録でダウンロード可能

# ダウンロードしたらcolabにアップロード
# 下記のコードの4行目open関数の第一引数に、文字列で動画ファイル名を入れて実行

from IPython.display import display, Javascript
from base64 import b64encode

binary = open(, "rb").read()
display(Javascript(f"""
  const video = document.createElement("video");
  video.height = 240;
  video.volume = 0;
  video.autoplay = true;
  video.controls = true;
  const source = document.createElement("source");
  source.src = "data:video/mp4;base64,{b64encode(binary).decode()}";
  video.appendChild(source);
  document.querySelector("#output-area").appendChild(video);
"""));

# 動画再生できたらこのセルは畳み、次セルに進む

# 注：Chapter 5 で動画再生用の独自関数 play_video を作りましたが、今回は不使用
#（実行のつど変換処理が走って遅いため）

In [ ]:
# @title (2) 動画からフレーム抽出 (Chapter6-2のおさらい)

# まずOpenCVと画像表示用の独自関数 colab_imshow を準備し、
# OpenCVで動画を読み込み、1フレームずつ抽出して表示
# 動画を読み込む VideoCapture メソッドの引数に、文字列で動画ファイル名を入れて実行

# 表示はカクカクして遅いです (その理由は？)
# 動画ファイル名を入れ忘れると、エラーにならないが何も表示されない

# 途中でセル実行の中止ボタンを押せば、エラーなく中止できます (try..except構文を利用)
# 動作確認できたらこのセルは畳み、次セルに進む

import cv2
import requests

branch = "https://github.com/ec22s/colab-ikinari-python/raw/refs/heads/main"
paths = [ "util/colab_imshow.py" ]
for path in paths:
  exec(requests.get(f"{branch}/{path}", allow_redirects=True).content)

cap = cv2.VideoCapture("17803_1280x720.mp4") # この引数に動画ファイル名を入れる

try:
  while cap.isOpened():
    # フレームを抽出
    ret, frame = cap.read()

    if ret:
      # 画像を表示
      colab_imshow("png", frame)

except KeyboardInterrupt:
  print("中止しました")

In [ ]:
# @title (3) 物体検出ライブラリを準備 (Chapter6-2のおさらい)

# 本 p.181 のライブラリインストールのColab版です
# 1回実行すればよいので(ランタイムを再起動しない限り)、先にこれだけ実行しておきます
# 実行したらこのセルは畳み、次セルに進む

import subprocess
subprocess.run(["pip", "install", "ultralytics"])

In [ ]:
# @title (4) 動画から人を検出、人数を表示 (Chapter6-2のおさらい)

# Chapter6-2 (6)〜(10) の動画を差し替えただけです
# 学習済みの人は、コードをコピーし動画ファイル名を差し替えればOK

# 未習の人は、下記の(6)〜(10)を順番に行って動かします
# https://colab.research.google.com/github/ec22s/colab-ikinari-python/blob/main/base/base_chapter_6_2.ipynb

# 完成例は下記の(10)
# https://colab.research.google.com/github/ec22s/colab-ikinari-python/blob/main/completed/completed_chapter_6_2.ipynb

# 参考まで、完成例の概要を書いておきます

・ライブラリのインポート (YOLO, cv2, numpy)

・テキストを画像に描画する関数 text_overwrite_to_image を定義

・物体検出のモデル読み込み
・動画読み込み

・動画の1フレームずつを画像として抽出し、物体検出を行い、人数を画像の上部に追加表示

# 動画のサイズにより、画像上部の人数表示の大きさを調整すべきかも
# その際に変える数値パラメータは下記3箇所

extension_height = 120
font_scale = 2
img_annotated = text_overwrite_to_image(..., (50, 75))

# 動作確認できたらこのセルは畳み、次セルに進む

In [ ]:
# @title (5) 顔の領域を抽出 (本 6-5-1)

# ここから新しいコードを書いていきます

# 一つ前のセルのコードを、このセルにコピー
# (時間がなければ既存コードに追加でも可)
# 次に、本のコード 6-5-1 (p.195〜196) と同様に以下4点を追加

# 1. 顔を検出・描画する関数 detect_faces
# 2. 顔の検出モデル face_cascade
# 3. 人を検出した領域のリスト person_boxes
# 4. person_boxesの各領域をforループで関数detect_facesに渡す繰り返し処理

# タイプ量が多く間違いやすいので、ゆっくり焦らず書く (入力補完も活用して)

# 顔が検出されない(にくい)時は本 p.200 のとおりパラメータを調整
# 問題なければこのセルは畳み、次セルに進む

# ↓ 以下にコードを書く


In [ ]:
# @title (6) 笑顔を検出 (本 6-5-2)

# 一つ前のセルのコードを、このセルにコピー
# (時間がなければ既存コードに追加でも可)
# 次に、本のコード 6-5-2, 6-5-3 (p.198〜199) と同様に以下3点を追加

# 1. 笑顔(のパーツ)を検出するモデル smile_cascade
# 2. detect_faces関数の引数にsmile_cascadeを追加 (使用場所と関数定義の両方)
# 3. 笑顔を検出する処理 (顔を検出するforループ内)

# 追加部分を単純にタイプするだけでなく、似たような箇所をコピーして省力化しても良い

# 顔・笑顔が検出されない(にくい)時は本 p.200 のとおりパラメータを調整
# 問題なければこのセルは畳み、次セル (今回の最後) に進む

# ↓ 以下にコードを書く


In [ ]:
# @title (7) 笑顔の人数を表示 (本 6-5-3)

# 一つ前のセルのコードを、このセルにコピー
# (時間がなければ既存コードに追加でも可)
# 次に、本のコード 6-5-4 (p.201) と同様に以下を追加

# 1. 顔を検出する関数 detect_faces の中に、笑顔フラグの変数 is_smile を作成し戻り値に追加
# 2. 各フレームの処理に笑顔の数を格納する変数 count_smile を追加し、detect_faces の戻り値
# の is_smile が真なら +1 (インクリメントと言う)
# 3. 画像を表示する直前に、text_overwrite_to_image の実行を挿入して笑顔の人数を表示

# 追加する場所があちこちにあるので、漏れがないよう本と見比べて注意

# 顔・笑顔が検出されない(にくい)時は本 p.200 のとおりパラメータを調整
# 実行して本の p.202 のようになればOK

# ↓ 以下にコードを書く


In [ ]:
# @title (8) 全員笑顔になったら撮影 (本 6-6-1)

# ここでの「撮影」は、動画内のフレームを画像として保存すること

# 一つ前のセルのコードを、このセルにコピー
# (時間がなければ既存コードに追加でも可)
# 次に、本のコード 6-6-1 (p.203) と同様に以下を追加

# 1. パッケージ2つ
# 2. 画像保存の設定
# 3. 人数と笑顔数が一致したら画像を保存する処理

# 実行して本の p.205 のように全員笑顔の画像が保存されていればOK

# ↓ 以下にコードを書く


In [ ]:
# @title (9) 応用例：全員笑顔になったら動画を止めて撮影、文字と元画像 (少し大きく) を表示

# 時間があれば(8)の応用例として「全員笑顔になった時」の処理を追加

# ① 動画を止める（whileループを抜ける）
# ② 出力欄に、全員笑顔になった旨を表示
# ③ 元画像を少し大きく表示（関数 colab_imshow の第三引数に画像の高さを渡す）

# ↓ 以下にコードを書く


In [ ]:
# @title これで学習会は終わりです。お疲れ様でした！